<em style="text-align:center">Copyright Iván Pinar Domínguez</em>

## Importar librerías iniciales e instancia de modelo de chat

In [1]:
from langchain_core.prompts import PromptTemplate, SystemMessagePromptTemplate,ChatPromptTemplate, HumanMessagePromptTemplate
#from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_classic.chains import SimpleSequentialChain, LLMChain,TransformChain
#f = open('../OpenAI_key.txt')
#api_key = f.read()
#llm = ChatOpenAI(openai_api_key=api_key)
llm = ChatOllama(model='gpt-oss:20b-cloud')

## Importamos documentos

In [2]:
from langchain_community.document_loaders import WikipediaLoader

In [ ]:
consulta_wikipedia = input("Escribe tu consulta para Wikipedia: ")

In [4]:
idioma_final = input()

 alemán


In [5]:
loader = WikipediaLoader(query=consulta_wikipedia,lang="es",load_max_docs=10)

In [6]:
data = loader.load()

C:\Users\Iván Pinar\anaconda3\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file C:\Users\Iván Pinar\anaconda3\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [7]:
data[0].page_content

'¡Hola! es una revista semanal española especializada en noticias de celebridades, publicada en Barcelona, España, y en otros quince países, con ediciones locales en Argentina, Brasil, Canadá, Chile, Colombia, Filipinas, Estados Unidos, Grecia, Indonesia, México, Pakistán, Perú, Puerto Rico, Reino Unido, Tailandia y Venezuela. Es la segunda revista española más vendida después de Pronto.\u200b\n\n\n== Historia ==\nEl 11 de septiembre de 1944, se sacó a la venta el primer número de la revista\u200b fundada y dirigida por Antonio Sánchez y Mercedes Junco.\nDesde 1984 fue su hijo, Eduardo Sánchez Junco, quien les sucedió en la dirección\u200b hasta su fallecimiento el 14 de julio de 2010 tras una duradera enfermedad.\u200b Momento desde el cual asumen la dirección sus tres hijos: Mercedes Sánchez Pérez, Mamen Sánchez Pérez y Eduardo Sánchez Pérez. \nEn 1989 se lanzó la versión inglesa del semanario en el Reino Unido. Y, desde 1998, también se editó una versión francesa llamada Oh la!, que

In [8]:
texto_entrada = data[0].page_content

# TransformChain

### Definir la función de transformación personalizada

In [9]:
def transformer_function(inputs: dict) -> dict: #Toma de entrada un diccionario y lo devuelve con la transformación oportuna
    texto = inputs['texto']
    primer_parrafo = texto.split('\n')[0]
    return {'salida':primer_parrafo}

In [10]:
transform_chain = TransformChain(input_variables=['texto'],
                                 output_variables=['salida'],
                                 transform=transformer_function)

## Definir la cadena secuencial

In [11]:
#Creamos bloque LLMChain para resumir
template1 = "Crea un resumen en una línea del siguiente texto:\n{texto}"
prompt = ChatPromptTemplate.from_template(template1)
summary_chain = LLMChain(llm=llm,
                     prompt=prompt,
                     output_key="texto_resumen")

C:\Users\Iván Pinar\AppData\Local\Temp\ipykernel_14848\2019573838.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  summary_chain = LLMChain(llm=llm,


In [12]:
#Creamos bloque LLMChain para traducir
template2 = "Traduce a"+ idioma_final + "el siguiente texto:\n{texto}"
prompt = ChatPromptTemplate.from_template(template2)
#prompt.format_prompt(idioma=idioma_final)
translate_chain = LLMChain(llm=llm,
                     prompt=prompt,
                     output_key="texto_traducido")

In [13]:
sequential_chain = SimpleSequentialChain(chains=[transform_chain,summary_chain,translate_chain],
                                        verbose=True)

In [14]:
result = sequential_chain(texto_entrada)

C:\Users\Iván Pinar\AppData\Local\Temp\ipykernel_14848\2789050324.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = sequential_chain(texto_entrada)




> Entering new SimpleSequentialChain chain...
¡Hola! es una revista semanal española especializada en noticias de celebridades, publicada en Barcelona, España, y en otros quince países, con ediciones locales en Argentina, Brasil, Canadá, Chile, Colombia, Filipinas, Estados Unidos, Grecia, Indonesia, México, Pakistán, Perú, Puerto Rico, Reino Unido, Tailandia y Venezuela. Es la segunda revista española más vendida después de Pronto.​
Revista ¡Hola!: líder en noticias de celebridades, publicada en 15 países.
Revista ¡Hola!: líder en noticias de celebridades, veröffentlicht in 15 Ländern.

> Finished chain.
